In [ ]:
# ==============================================================================
# Script ETL Completo - Versão Atualizada com Extração de is_fragil, degelo, prioridade_alocacao e peso
# ==============================================================================
# Este script extrai e combina dados de múltiplas abas da Planilha Google,
# inclui is_fragil, degelo, prioridade_alocacao e calcula peso a partir do nome do produto.
# Ao final, grava os resultados na aba "MIX" e gera logs em novas abas na mesma planilha.
# ==============================================================================

# ==============================================================================
# Bloco 1: Instalação e Importação de Bibliotecas
# ==============================================================================
# !pip install openpyxl gspread gspread-dataframe -q  # Descomente se necessário

import pandas as pd
import numpy as np
import re
import sys
try:
    import gspread
    import gspread_dataframe as gd
except Exception:
    gspread = None
    gd = None

try:
    from google.colab import auth
    from google.auth import default
except Exception:
    auth = None
    default = None

print("✅ Ambiente preparado e bibliotecas importadas com sucesso!")
print("   - pandas, numpy, gspread, gspread_dataframe, google.colab.auth, google.auth.default")


# ==============================================================================
# Bloco 2: Configuração Principal do Script
# ==============================================================================
# ATENÇÃO: Este é o painel de controle do script. Os nomes das abas
# que o script procurará na sua Planilha Google estão definidos aqui.

NOMES_DAS_ABAS = {
    "mix": "Mix",
    "volumetria": "volumetria e fabricantes",
    "categorias_gpt": "Categoria ChatGPT",
    "categorias_site": "Categoria Site",
    "vendas": "Vendas Pamplona",
    "subcategorias": "Subcategorias",
    "config_operacionais": "Configuracoes_Operacionais",
    "volumetria_equipamentos": "Volumetria_Equipamentos",
    "caixaria": "Caixaria",
    "caixaria_nova_compras": "Caixaria nova compras",
    "volumes_secundarios": "Volumes secundários",
}

# --- Modo local (XLSX) ---
USE_LOCAL_XLSX = False
LOCAL_XLSX_PATH = "ETL [PINHEIROS] (5).xlsx"
LOCAL_OUTPUT_XLSX = "ETL_OUTPUT.xlsx"
AUTO_SKIP_VOLUME_PROMPT = True
NOME_PLANILHA_PRINCIPAL = ""  # coloque o NOME exato da planilha para evitar prompt


# Aba opcional de vendas alvo (Pinheiros)
NOME_ABA_VENDAS_ALVO = "vendas alvo"
limite_peso_kg = None  # sera preenchido a partir de Configuracoes_Operacionais

# Planilha de produtos geradores (is_fragil, degelo, prioridade_alocacao)
NOME_PLANILHA_PRODUTOS_GERADORES = "Produtos geradores vila olimpia"
ID_PLANILHA_PRODUTOS_GERADORES = "12sypXkRv0_YesWE66bT7htmmXYxNpHu6Mp_Y4rt5fGo"
ABAS_PRODUTOS_GERADORES = {
    "degelo_nao": "NÃO PODE SOFRER DEGELO",
    "degelo_pode": "PODE SOFRER DEGELO"
}

# Aba de saída principal
NOME_ABA_SAIDA_MIX = "MIX ENRIQUECIDO"

print("✅ Configurações de nomes de abas definidas.")
print("   - Aba de saída principal: 'MIX'.")
print("   - Planilha de produtos geradores configurada.")


# ==============================================================================
# Bloco 3: Autenticação e Abertura da Planilha Principal
# ==============================================================================
print("\n--- [ Etapa 1 de 6 ]: Autenticação e Abertura da Planilha Principal ---")
if USE_LOCAL_XLSX:
    print(f"PASSO 1: Modo local ativado. Usando XLSX: '{LOCAL_XLSX_PATH}'.")
    spreadsheet = pd.ExcelFile(LOCAL_XLSX_PATH)
    gc = None
else:
    if auth is None or default is None or gspread is None:
        print("❌ ERRO: Bibliotecas de Google não disponíveis. Instale gspread e execute no Colab, ou use USE_LOCAL_XLSX=True.")
        sys.exit()
    print("PASSO 1: Autenticando com a conta Google...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("✅ Autenticação concluída.")


def _to_float(valor):
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return None
    if isinstance(valor, (int, float)):
        return float(valor)
    s = str(valor).strip()
    if not s:
        return None
    s = s.replace(',', '.')
    try:
        return float(s)
    except Exception:
        return None




def _normalizar_caixaria(df):
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    if len(df.columns) == 0:
        return pd.DataFrame()
    col_code = None
    for c in df.columns:
        if c in ['cod_produto', 'codigo_produto', 'product_code']:
            col_code = c
            break
    if col_code is None:
        col_code = df.columns[0]
    col_caix = None
    for c in df.columns:
        if 'caixaria' in c:
            col_caix = c
            break
    if col_caix is None:
        col_caix = df.columns[-1] if len(df.columns) > 1 else df.columns[0]
    df = df[[col_code, col_caix]].copy()
    df.rename(columns={col_code: 'cod_produto', col_caix: 'caixaria'}, inplace=True)
    df['cod_produto'] = df['cod_produto'].astype(str).str.strip().str.upper()
    df['caixaria'] = pd.to_numeric(df['caixaria'].astype(str).str.replace(',', '.'), errors='coerce')
    df = df.dropna(subset=['cod_produto'])
    df = df.drop_duplicates(subset='cod_produto', keep='first')
    return df


def _normalizar_volumes_secundarios(df):
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    if len(df.columns) == 0:
        return pd.DataFrame()
    col_code = None
    for c in df.columns:
        if c in ['cod_produto', 'codigo_produto', 'product_code']:
            col_code = c
            break
    if col_code is None:
        col_code = df.columns[0]

    def _find_col(keys):
        for c in df.columns:
            for k in keys:
                if k in c:
                    return c
        return None

    col_l = _find_col(['caixa_largura', 'largura_caixa'])
    col_a = _find_col(['caixa_altura', 'altura_caixa'])
    col_c = _find_col(['caixa_comprimento', 'comprimento_caixa'])
    col_v = _find_col(['caixa_volume', 'volume_caixa'])

    cols = [c for c in [col_code, col_l, col_a, col_c, col_v] if c is not None]
    df = df[cols].copy()

    rename = {col_code: 'cod_produto'}
    if col_l:
        rename[col_l] = 'caixa_largura_cm'
    if col_a:
        rename[col_a] = 'caixa_altura_cm'
    if col_c:
        rename[col_c] = 'caixa_comprimento_cm'
    if col_v:
        rename[col_v] = 'caixa_volume_cm3'
    df.rename(columns=rename, inplace=True)

    df['cod_produto'] = df['cod_produto'].astype(str).str.strip().str.upper()
    for col in ['caixa_largura_cm', 'caixa_altura_cm', 'caixa_comprimento_cm', 'caixa_volume_cm3']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')

    df = df.dropna(subset=['cod_produto'])
    df = df.drop_duplicates(subset='cod_produto', keep='first')
    return df
def _normalizar_tipo_equip(valor):
    return str(valor).strip().lower().replace(' ', '_')

def _calc_escaninhos(vol_l_total, capacidade):
    try:
        if vol_l_total is None or (isinstance(vol_l_total, float) and np.isnan(vol_l_total)):
            return 1
        if capacidade is None or capacidade <= 0:
            return 1
        vol = float(vol_l_total)
        if vol <= 0:
            return 1
        return int(np.ceil(vol / capacidade))
    except Exception:
        return 1

def _buscar_limite_peso_kg(df_config):
    if df_config is None or df_config.empty:
        return None
    df = df_config.copy()
    cols_lower = {str(c).strip().lower(): c for c in df.columns}

    # Caso 1: coluna direta com o limite
    for key in ['limite_peso_kg', 'limite_peso', 'peso_limite_kg', 'limite_peso_kg_unitario']:
        if key in cols_lower:
            serie = df[cols_lower[key]].dropna()
            if not serie.empty:
                return _to_float(serie.iloc[0])

    # Caso 2: estrutura de parametros (parametro/valor)
    param_col = None
    for key in ['parametro', 'parâmetro', 'chave', 'configuracao', 'configuração', 'nome', 'key']:
        if key in cols_lower:
            param_col = cols_lower[key]
            break
    value_col = None
    for key in ['valor', 'value', 'val']:
        if key in cols_lower:
            value_col = cols_lower[key]
            break
    if param_col and value_col:
        serie = df[param_col].astype(str).str.strip().str.lower()
        mask = serie == 'limite_peso_kg'
        if not mask.any():
            mask = serie.str.replace(' ', '_').str.contains('limite') & serie.str.contains('peso')
        if mask.any():
            return _to_float(df.loc[mask, value_col].iloc[0])

    return None

if USE_LOCAL_XLSX:
    nome_planilha_principal = LOCAL_XLSX_PATH
    print(f"\nPASSO 2: Usando planilha local: '{nome_planilha_principal}'.")
else:
    try:
        if NOME_PLANILHA_PRINCIPAL:
            nome_planilha_principal = NOME_PLANILHA_PRINCIPAL
        else:
            nome_planilha_principal = input("\n➡️  Digite o NOME EXATO da sua Planilha Google principal e pressione Enter: ")
        print(f"\nPASSO 2: Abrindo a planilha: '{nome_planilha_principal}'...")
        spreadsheet = gc.open(nome_planilha_principal)
        print("✅ Planilha encontrada!")
    except Exception:
        print(f"\n❌ ERRO CRÍTICO: A planilha '{nome_planilha_principal}' não foi encontrada ou você não tem acesso.")
        sys.exit()


# ==============================================================================
# Bloco 4: Carregamento, Validação e Limpeza dos Dados (Planilha Principal)
# ==============================================================================
print("\n--- [ Etapa 2 de 6 ]: Carregando e Validando Dados ---")

if USE_LOCAL_XLSX:
    titulos_planilha = spreadsheet.sheet_names
else:
    titulos_planilha = [ws.title for ws in spreadsheet.worksheets()]
titulos_lower = {str(t).strip().lower(): t for t in titulos_planilha}

vendas_alvo_title = titulos_lower.get(str(NOME_ABA_VENDAS_ALVO).strip().lower())
vendas_pamplona_title = titulos_lower.get(str(NOMES_DAS_ABAS["vendas"]).strip().lower())
config_operacionais_title = titulos_lower.get(str(NOMES_DAS_ABAS.get('config_operacionais', 'Configuracoes_Operacionais')).strip().lower())
volumetria_equip_title = titulos_lower.get(str(NOMES_DAS_ABAS.get('volumetria_equipamentos', 'Volumetria_Equipamentos')).strip().lower())
caixaria_title = titulos_lower.get(str(NOMES_DAS_ABAS.get('caixaria', 'Caixaria')).strip().lower())
caixaria_nova_title = titulos_lower.get(str(NOMES_DAS_ABAS.get('caixaria_nova_compras', 'Caixaria nova compras')).strip().lower())
volumes_sec_title = titulos_lower.get(str(NOMES_DAS_ABAS.get('volumes_secundarios', 'Volumes secundários')).strip().lower())

# Abas obrigatórias (vendas pode ser 'vendas alvo' ou 'Vendas Pamplona')
abas_obrigatorias = [
    NOMES_DAS_ABAS["mix"],
    NOMES_DAS_ABAS["volumetria"],
    NOMES_DAS_ABAS["categorias_gpt"],
    NOMES_DAS_ABAS["categorias_site"],
    NOMES_DAS_ABAS["subcategorias"],
]
abas_faltando = [nome_real for nome_real in abas_obrigatorias if nome_real not in titulos_planilha]

tem_vendas_alvo = vendas_alvo_title is not None
tem_vendas_pamplona = vendas_pamplona_title is not None

if abas_faltando or (not tem_vendas_alvo and not tem_vendas_pamplona):
    faltantes_msg = abas_faltando[:]
    if not tem_vendas_alvo and not tem_vendas_pamplona:
        faltantes_msg.append(f"{NOME_ABA_VENDAS_ALVO} ou {NOMES_DAS_ABAS['vendas']}")
    print(f"\n❌ ERRO CRÍTICO: As seguintes abas obrigatórias não foram encontradas: {faltantes_msg}.")
    sys.exit()

print("✅ Todas as abas necessárias foram encontradas.")
if tem_vendas_alvo:
    print(f"✅ Aba de vendas alvo detectada: '{vendas_alvo_title}'.")
else:
    print(f"✅ Usando aba de vendas padrão: '{vendas_pamplona_title}'.")





def _carregar_aba(spreadsheet_obj, nome_aba):
    if USE_LOCAL_XLSX:
        df = pd.read_excel(LOCAL_XLSX_PATH, sheet_name=nome_aba)
    else:
        df = gd.get_as_dataframe(spreadsheet_obj.worksheet(nome_aba), evaluate_formulas=True)
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _normalizar_vendas_alvo(df):
    df = df.copy()
    rename_map = {}
    for col in df.columns:
        col_norm = str(col).strip().lower()
        if col_norm in ['dt_previsao_entrega', 'data_previsao_entrega', 'dt_prevista_entrega', 'data_entrega']:
            rename_map[col] = 'dt_previsao_entrega'
        elif col_norm in ['cod_produto', 'codigo_produto', 'product_code', 'codproduto']:
            rename_map[col] = 'cod_produto'
        elif col_norm in ['desc_produto', 'descricao_produto', 'produto', 'product_name']:
            rename_map[col] = 'desc_produto'
        elif 'qtd_total' in col_norm and 'sum' in col_norm:
            rename_map[col] = 'qtd_total'
        elif col_norm == 'qtd_total':
            rename_map[col] = 'qtd_total'
    if rename_map:
        df.rename(columns=rename_map, inplace=True)
    if 'cod_produto' in df.columns:
        df['cod_produto'] = df['cod_produto'].astype(str).str.strip()
    if 'desc_produto' in df.columns:
        df['desc_produto'] = df['desc_produto'].astype(str).str.strip()
    if 'qtd_total' in df.columns:
        df['qtd_total'] = pd.to_numeric(df['qtd_total'].astype(str).str.replace(',', '.'), errors='coerce')
    if 'dt_previsao_entrega' in df.columns:
        df['dt_previsao_entrega'] = pd.to_datetime(df['dt_previsao_entrega'], errors='coerce')
    return df

try:
    print("Iniciando o carregamento das abas da planilha principal...")
    df_mix = _carregar_aba(spreadsheet, NOMES_DAS_ABAS["mix"])
    df_vol = _carregar_aba(spreadsheet, NOMES_DAS_ABAS["volumetria"])
    df_categoria_gpt = _carregar_aba(spreadsheet, NOMES_DAS_ABAS["categorias_gpt"])
    df_categoria_site = _carregar_aba(spreadsheet, NOMES_DAS_ABAS["categorias_site"])
    if tem_vendas_alvo:
        df_vendas = _carregar_aba(spreadsheet, vendas_alvo_title)
        df_vendas = _normalizar_vendas_alvo(df_vendas)
    else:
        df_vendas = _carregar_aba(spreadsheet, vendas_pamplona_title)

    df_subcategorias = _carregar_aba(spreadsheet, NOMES_DAS_ABAS["subcategorias"])
    if volumetria_equip_title:
        df_vol_equip = _carregar_aba(spreadsheet, volumetria_equip_title)
        print(f"OK: Aba de volumetria de equipamentos carregada: '{volumetria_equip_title}'.")
    else:
        df_vol_equip = pd.DataFrame()
        print("WARNING: Aba Volumetria_Equipamentos nao encontrada. Escaninhos necessarios nao serao calculados.")
    if config_operacionais_title:
        df_config_operacionais = _carregar_aba(spreadsheet, config_operacionais_title)
        print(f"✅ Aba de configuracoes operacionais carregada: '{config_operacionais_title}'.")
    else:
        df_config_operacionais = pd.DataFrame()
        print("⚠️  Aba Configuracoes_Operacionais não encontrada. Usando limite padrão para peso.")

    # Abas opcionais de caixaria e volumes secundarios
    try:
        if caixaria_title:
            df_caixaria = _carregar_aba(spreadsheet, caixaria_title)
            print(f"✅ Aba de caixaria carregada: '{caixaria_title}'.")
        else:
            df_caixaria = pd.DataFrame()
            print("⚠️  Aba Caixaria não encontrada.")
    except Exception as e:
        df_caixaria = pd.DataFrame()
        print(f"⚠️  Falha ao carregar aba Caixaria: {e}")

    try:
        if caixaria_nova_title:
            df_caixaria_nova = _carregar_aba(spreadsheet, caixaria_nova_title)
            print(f"✅ Aba de caixaria nova compras carregada: '{caixaria_nova_title}'.")
        else:
            df_caixaria_nova = pd.DataFrame()
            print("⚠️  Aba Caixaria nova compras não encontrada.")
    except Exception as e:
        df_caixaria_nova = pd.DataFrame()
        print(f"⚠️  Falha ao carregar aba Caixaria nova compras: {e}")

    try:
        if volumes_sec_title:
            df_volumes_sec = _carregar_aba(spreadsheet, volumes_sec_title)
            print(f"✅ Aba de volumes secundarios carregada: '{volumes_sec_title}'.")
        else:
            df_volumes_sec = pd.DataFrame()
            print("⚠️  Aba Volumes secundários não encontrada.")
    except Exception as e:
        df_volumes_sec = pd.DataFrame()
        print(f"⚠️  Falha ao carregar aba Volumes secundários: {e}")
    print("✅ Todas as abas necessárias foram carregadas com sucesso!")

    # Limpeza simples de colunas numéricas comuns
    if 'Quantidade' in df_mix.columns:
        df_mix['Quantidade'] = pd.to_numeric(df_mix['Quantidade'].astype(str).str.replace(',', '.'), errors='coerce')
    for col in ['largura_cm', 'altura_cm', 'comprimento_cm', 'volume_cm3']:
        if col in df_vol.columns:
            df_vol[col] = pd.to_numeric(df_vol[col].astype(str).str.replace(',', '.'), errors='coerce')

    for col in ['qtd_total', 'preco_unitario']:
        if col in df_vendas.columns:
            df_vendas[col] = pd.to_numeric(df_vendas[col].astype(str).str.replace(',', '.'), errors='coerce')

    if 'df_vol_equip' in locals() and df_vol_equip is not None and not df_vol_equip.empty:
        for col in ['l_por_escaninho', 'fator_seguranca']:
            if col in df_vol_equip.columns:
                df_vol_equip[col] = pd.to_numeric(df_vol_equip[col].astype(str).str.replace(',', '.'), errors='coerce')

    if 'id_modelo' in df_vol.columns:
        df_vol.drop(columns=['id_modelo'], inplace=True)
        print("  - [LIMPEZA] A coluna 'id_modelo' foi removida da base de volumetria.")

except Exception as e:
    print(f"\n❌ Ocorreu um erro inesperado ao ler as abas: {e}")
    sys.exit()

# Leitura do limite de peso (Configuracoes_Operacionais)
limite_peso_kg = _buscar_limite_peso_kg(df_config_operacionais) if 'df_config_operacionais' in locals() else None
if limite_peso_kg is None:
    limite_peso_kg = 0.7
    print(f"⚠️  limite_peso_kg não encontrado na aba Configuracoes_Operacionais. Usando padrão: {limite_peso_kg} kg.")
else:
    print(f"✅ limite_peso_kg carregado: {limite_peso_kg} kg.")



# ==============================================================================
# Bloco 4.1: Carregamento da Planilha de Produtos Geradores (automático)
# ==============================================================================
print("\n--- Carregando planilha de produtos geradores ---")

df_produtos_geradores = pd.DataFrame()

if USE_LOCAL_XLSX:
    print("  - [AVISO] Modo local: ignorando planilha de produtos geradores.")
else:

    def _abrir_planilha_produtos_geradores(gc_client):
        try:
            return gc_client.open_by_key(ID_PLANILHA_PRODUTOS_GERADORES)
        except Exception:
            return gc_client.open(NOME_PLANILHA_PRODUTOS_GERADORES)

    try:
        spreadsheet_produtos = _abrir_planilha_produtos_geradores(gc)
        print(f"  - [OK] Planilha de produtos geradores encontrada: '{spreadsheet_produtos.title}'.")

        try:
            df_degelo_nao = _carregar_aba(spreadsheet_produtos, ABAS_PRODUTOS_GERADORES["degelo_nao"])
            print(f"  - [OK] Aba '{ABAS_PRODUTOS_GERADORES['degelo_nao']}' carregada.")
        except Exception as e:
            print(f"  - [AVISO] Não foi possível carregar a aba '{ABAS_PRODUTOS_GERADORES['degelo_nao']}': {e}")
            df_degelo_nao = pd.DataFrame()

        try:
            df_degelo_pode = _carregar_aba(spreadsheet_produtos, ABAS_PRODUTOS_GERADORES["degelo_pode"])
            print(f"  - [OK] Aba '{ABAS_PRODUTOS_GERADORES['degelo_pode']}' carregada.")
        except Exception as e:
            print(f"  - [AVISO] Não foi possível carregar a aba '{ABAS_PRODUTOS_GERADORES['degelo_pode']}': {e}")
            df_degelo_pode = pd.DataFrame()

        if not df_degelo_nao.empty or not df_degelo_pode.empty:
            df_produtos_geradores = pd.concat([df_degelo_nao, df_degelo_pode], ignore_index=True)
            if 'product_code' in df_produtos_geradores.columns:
                df_produtos_geradores['product_code'] = df_produtos_geradores['product_code'].astype(str).str.strip().str.upper()
                if 'degelo' in df_produtos_geradores.columns:
                    df_produtos_geradores['degelo'] = df_produtos_geradores['degelo'].astype(str).str.strip().str.upper()
                    df_produtos_geradores['__has_degelo'] = df_produtos_geradores['degelo'].replace('NAN', '') != ''
                    df_produtos_geradores = df_produtos_geradores.sort_values('__has_degelo', ascending=False)
                    df_produtos_geradores.drop(columns='__has_degelo', inplace=True)
                df_produtos_geradores = df_produtos_geradores.drop_duplicates(subset='product_code', keep='first')
                print(f"  - [OK] Base de produtos geradores combinada com {len(df_produtos_geradores)} produtos únicos.")
            else:
                print("  - [AVISO] Coluna 'product_code' não encontrada na planilha de produtos geradores.")
                df_produtos_geradores = pd.DataFrame()
        else:
            print("  - [AVISO] Nenhuma aba de produtos geradores foi carregada com sucesso.")

    except Exception as e:
        print(f"  - [AVISO] Não foi possível conectar na planilha de produtos geradores: {e}")
        df_produtos_geradores = pd.DataFrame()


# ==============================================================================
# Bloco 5: Construção e Enriquecimento da Base de Dados
# ==============================================================================
print("\n--- [ Etapa 3 de 6 ]: Construindo a Base Unificada ---")

# --- 5a. Criação da Base a partir do Mix ---
if 'product_code' not in df_mix.columns or 'product_name' not in df_mix.columns or 'Quantidade' not in df_mix.columns:
    print("\n❌ ERRO CRÍTICO: A aba 'Mix' deve conter as colunas 'product_code', 'product_name' e 'Quantidade'.")
    sys.exit()

df_final = df_mix[['product_code', 'product_name', 'Quantidade']].copy()
df_final.rename(columns={'Quantidade': 'quantidade'}, inplace=True)
df_final['chave_juncao'] = df_final['product_code'].astype(str).str.strip().str.upper()
total_produtos_inicial = len(df_final)
print(f"  - Base inicial criada com {total_produtos_inicial} produtos a partir da aba '{NOMES_DAS_ABAS['mix']}'.")

# --- 5b. Funções de Junção (Merge) Modulares ---
def juntar_dados(df_base, df_para_juntar, chave_df, colunas_para_juntar, novos_nomes=None):
    if novos_nomes is None:
        novos_nomes = {}
    if chave_df not in df_para_juntar.columns:
        return df_base
    df_para_juntar = df_para_juntar.copy()
    df_para_juntar['chave_juncao'] = df_para_juntar[chave_df].astype(str).str.strip().str.upper()
    df_para_juntar.drop_duplicates(subset='chave_juncao', keep='first', inplace=True)
    df_base = pd.merge(df_base, df_para_juntar[['chave_juncao'] + colunas_para_juntar], on='chave_juncao', how='left')
    if novos_nomes:
        df_base.rename(columns=novos_nomes, inplace=True)
    return df_base

# --- 5c. Executando as Junções ---
print("  - Juntando dados de categorias, subcategorias e volumetria...")
df_final = juntar_dados(df_final, df_categoria_gpt, 'Cod_Produto', ['Categoria_Correta'], {'Categoria_Correta': 'categoria_armazenagem'})
df_final = juntar_dados(df_final, df_categoria_site, 'cod_produto', ['categoria'], {'categoria': 'categoria_site'})
df_final = juntar_dados(df_final, df_subcategorias, 'product_code', ['subcategoria'])
df_final = juntar_dados(df_final, df_vol, 'cod_produto', ['nm_fabricante', 'largura_cm', 'altura_cm', 'comprimento_cm', 'volume_cm3'], {'volume_cm3': 'volume_cm3_final'})

# --- 5c.1 Caixaria e Volumes Secundarios (opcional) ---
df_caixaria_final = pd.DataFrame()
try:
    df_caixaria_norm = _normalizar_caixaria(df_caixaria) if 'df_caixaria' in locals() else pd.DataFrame()
    df_caixaria_nova_norm = _normalizar_caixaria(df_caixaria_nova) if 'df_caixaria_nova' in locals() else pd.DataFrame()
    if not df_caixaria_norm.empty and not df_caixaria_nova_norm.empty:
        df_caixaria_final = pd.merge(df_caixaria_norm, df_caixaria_nova_norm, on='cod_produto', how='outer', suffixes=('_base', '_nova'))
        df_caixaria_final['caixaria'] = df_caixaria_final['caixaria_base'].combine_first(df_caixaria_final['caixaria_nova'])
        df_caixaria_final = df_caixaria_final[['cod_produto', 'caixaria']]
    elif not df_caixaria_norm.empty:
        df_caixaria_final = df_caixaria_norm
    elif not df_caixaria_nova_norm.empty:
        df_caixaria_final = df_caixaria_nova_norm
except Exception as e:
    print(f"  - [AVISO] Falha ao normalizar caixaria: {e}")
    df_caixaria_final = pd.DataFrame()
if not df_caixaria_final.empty:
    df_final = juntar_dados(df_final, df_caixaria_final, 'cod_produto', ['caixaria'])
    print(f"  - [OK] Caixaria juntada para {len(df_caixaria_final)} produtos.")
else:
    print("  - [AVISO] Nenhum dado de caixaria disponivel.")

try:
    df_vol_sec_norm = _normalizar_volumes_secundarios(df_volumes_sec) if 'df_volumes_sec' in locals() else pd.DataFrame()
except Exception as e:
    print(f"  - [AVISO] Falha ao normalizar volumes secundarios: {e}")
    df_vol_sec_norm = pd.DataFrame()
if not df_vol_sec_norm.empty:
    df_final = juntar_dados(df_final, df_vol_sec_norm, 'cod_produto', ['caixa_largura_cm', 'caixa_altura_cm', 'caixa_comprimento_cm', 'caixa_volume_cm3'])
    print(f"  - [OK] Volumes secundarios juntados para {len(df_vol_sec_norm)} produtos.")
else:
    print("  - [AVISO] Nenhum dado de volumes secundarios disponivel.")

# --- 5d. Produtos Geradores ---
if not df_produtos_geradores.empty:
    print("  - Juntando dados de produtos geradores (is_fragil, degelo, prioridade_alocacao)...")
    colunas_disponiveis = [c for c in ['is_fragil', 'degelo', 'prioridade_alocacao'] if c in df_produtos_geradores.columns]
    if colunas_disponiveis:
        df_final = juntar_dados(df_final, df_produtos_geradores, 'product_code', colunas_disponiveis)
        print(f"    - [OK] Colunas {', '.join(colunas_disponiveis)} extraídas e juntadas com sucesso.")
    else:
        print("    - [AVISO] Nenhuma das colunas esperadas foi encontrada na base de produtos geradores.")
else:
    print("  - [AVISO] Base de produtos geradores não disponível. is_fragil, degelo e prioridade_alocacao não serão extraídos.")

# --- 5d.1 Normalizacao de degelo ---
if 'degelo' in df_final.columns:
    df_final['degelo'] = df_final['degelo'].astype(str).str.strip().str.upper()
    df_final['degelo'] = df_final['degelo'].replace({'NAN': '', 'NONE': '', 'NULL': ''})
    df_final.loc[df_final['degelo'].isin(['NÃO', 'NAO', 'NÃO PODE', 'NAO PODE', 'NÃO PODE SOFRER DEGELO', 'NAO PODE SOFRER DEGELO']), 'degelo'] = 'NAO'
    df_final.loc[df_final['degelo'].isin(['PODE', 'PODE SOFRER DEGELO']), 'degelo'] = 'PODE'
    print("  - 'degelo' normalizado (PODE/NAO) e valores vazios limpos.")
else:
    print("  - [AVISO] Coluna 'degelo' nao encontrada para normalizacao.")

# --- 5e. Tratamento de Dados Pós-Junção e GERAÇÃO DE LOG ---
# Lógica para log de produtos sem VOLUMETRIA
df_log_volumetria = df_final[df_final[['nm_fabricante', 'altura_cm']].isnull().any(axis=1)].copy()
if not df_log_volumetria.empty:
    print(f"\n  - [ALERTA DE DADOS] Foram encontrados {len(df_log_volumetria)} produtos sem dados de fabricante ou altura.")
    print("    > Uma aba de log chamada 'Log_Produtos_Sem_Dados' será gerada.")
    df_log_volumetria = df_log_volumetria[['product_code', 'product_name', 'nm_fabricante', 'altura_cm']]
else:
    print("  - [OK] Todos os produtos do mix foram encontrados na base de volumetria.")

# Lógica para log de produtos sem SUBCATEGORIA
df_log_subcategoria = df_final[df_final['subcategoria'].isnull()].copy()
if not df_log_subcategoria.empty:
    print(f"  - [ALERTA DE DADOS] Foram encontrados {len(df_log_subcategoria)} produtos sem 'subcategoria' correspondente.")
    print("    > Uma aba de log chamada 'Log_Produtos_Sem_Subcategoria' será gerada.")
    df_log_subcategoria = df_log_subcategoria[['product_code', 'product_name']]
else:
    print("  - [OK] Todos os produtos do mix receberam uma subcategoria.")

# Cálculos de Qualidade de Dados
cat_armazenagem_faltantes = df_final['categoria_armazenagem'].isnull().sum()
perc_armazenagem = (cat_armazenagem_faltantes / total_produtos_inicial) * 100 if total_produtos_inicial > 0 else 0
subcat_faltantes = df_final['subcategoria'].isnull().sum()
perc_subcategoria = (subcat_faltantes / total_produtos_inicial) * 100 if total_produtos_inicial > 0 else 0

print(f"  - Análise de Categorias: {cat_armazenagem_faltantes} produtos ({perc_armazenagem:.2f}%) estão sem 'categoria_armazenagem'.")

# Avisos adicionais de qualidade
if cat_armazenagem_faltantes > 0:
    df_log_categoria_armazenagem = df_final[df_final['categoria_armazenagem'].isnull()].copy()
    print(f"  - [ALERTA DE DADOS] {len(df_log_categoria_armazenagem)} produtos estao sem 'categoria_armazenagem'.")
    df_log_categoria_armazenagem = df_log_categoria_armazenagem[['product_code', 'product_name']]
else:
    df_log_categoria_armazenagem = pd.DataFrame()
    print("  - [OK] Todos os produtos possuem 'categoria_armazenagem'.")

if 'degelo' in df_final.columns:
    mask_geladeira = df_final['categoria_armazenagem'].astype(str).str.lower().str.contains('refrigerado|geladeira', na=False)
    mask_degelo_vazio = df_final['degelo'].isnull() | (df_final['degelo'].astype(str).str.strip() == '')
    df_log_degelo_geladeira = df_final[mask_geladeira & mask_degelo_vazio].copy()
    if not df_log_degelo_geladeira.empty:
        print(f"  - [ALERTA DE DADOS] {len(df_log_degelo_geladeira)} produtos de geladeira estao com 'degelo' vazio.")
        df_log_degelo_geladeira = df_log_degelo_geladeira[['product_code', 'product_name', 'categoria_armazenagem', 'degelo']]
    else:
        print("  - [OK] Nenhum produto de geladeira com 'degelo' vazio.")
else:
    df_log_degelo_geladeira = pd.DataFrame()
    print("  - [AVISO] Coluna 'degelo' nao encontrada para checar geladeira.")

if 'volume_cm3_final' in df_final.columns:
    df_final['volume_cm3_final'].fillna(0, inplace=True)
    produtos_sem_volume = df_final[df_final['volume_cm3_final'] == 0]
    if not produtos_sem_volume.empty:
        num_sem_volume = len(produtos_sem_volume)
        print(f"\n  [ATENÇÃO] Foram detectados {num_sem_volume} produtos com volume informado igual a 0.")
        if AUTO_SKIP_VOLUME_PROMPT:
            print("  - [AVISO] AUTO_SKIP_VOLUME_PROMPT ativo. Mantendo volumes '0'.")
        else:
            while True:
                try:
                    r = input("  > Deseja substituir os volumes '0' por um valor padrão? (s/n): ").lower()
                    if r == 's':
                        valor_padrao_str = input("    > Digite um valor padrão em cm³ (ex: 1000): ")
                        valor_padrao = float(valor_padrao_str)
                        df_final.loc[df_final['volume_cm3_final'] == 0, 'volume_cm3_final'] = valor_padrao
                        print(f"  ✅ Volumes '0' foram substituídos por '{valor_padrao} cm³'.")
                        break
                    if r == 'n':
                        print("  - OK. Os volumes '0' serão mantidos.")
                        break
                except (ValueError, TypeError):
                    print("  ❌ Valor inválido. Por favor, digite apenas números.")
    else:
        print("  - ✅ Todos os produtos possuem dados de volumetria válidos.")

print("✅ Base de dados unificada com sucesso.")


# ==============================================================================
# Bloco 6: Análise de Vendas (Curva ABC)
# ==============================================================================
print("\n--- [ Etapa 4 de 6 ]: Calculando Curva ABC e Métricas de Vendas ---")

base_analise = 'qtd_total'
if 'preco_unitario' in df_vendas.columns and df_vendas['preco_unitario'].notna().any():
    df_vendas['valor_total'] = df_vendas['qtd_total'] * df_vendas['preco_unitario']
    base_analise = 'valor_total'
    print("  - Base da Curva ABC: VALOR (R$)")
else:
    print("  - Base da Curva ABC: VOLUME (unidades vendidas)")

# Garante que qtd_total esteja numérico
if 'qtd_total' in df_vendas.columns:
    df_vendas['qtd_total'] = pd.to_numeric(df_vendas['qtd_total'], errors='coerce')

# Determinar chave de vendas (prioriza código do produto quando existir)
if 'cod_produto' in df_vendas.columns and df_vendas['cod_produto'].notna().any():
    chave_vendas_col = 'cod_produto'
    print("  - Curva e vendas serão calculadas por código do produto.")
elif 'desc_produto' in df_vendas.columns:
    chave_vendas_col = 'desc_produto'
    print("  - Curva e vendas serão calculadas por descrição do produto.")
else:
    print("\n❌ ERRO CRÍTICO: A aba de vendas deve conter 'cod_produto' ou 'desc_produto'.")
    sys.exit()

df_vendas['chave_vendas'] = df_vendas[chave_vendas_col].astype(str).str.strip().str.upper()

df_analise_curva = df_vendas.groupby('chave_vendas').agg(total_relevancia=(base_analise, 'sum')).reset_index().sort_values(by='total_relevancia', ascending=False)
df_analise_curva['acumulado_%'] = (df_analise_curva['total_relevancia'] / df_analise_curva['total_relevancia'].sum()).cumsum()
df_analise_curva['curva'] = pd.cut(df_analise_curva['acumulado_%'], bins=[0, 0.8, 0.95, 1.01], labels=['A', 'B', 'C'], right=False)

if 'dt_previsao_entrega' in df_vendas.columns:
    df_vendas['dt_previsao_entrega'] = pd.to_datetime(df_vendas['dt_previsao_entrega'], errors='coerce')
    df_vendas.dropna(subset=['dt_previsao_entrega'], inplace=True)
    periodo_dias = (df_vendas['dt_previsao_entrega'].max() - df_vendas['dt_previsao_entrega'].min()).days + 1
    if periodo_dias <= 0:
        periodo_dias = 1
else:
    periodo_dias = 1

vendas_agregado = df_vendas.groupby('chave_vendas').agg(venda_total=('qtd_total', 'sum')).reset_index()
vendas_agregado['venda_media_diaria'] = vendas_agregado['venda_total'] / periodo_dias
print(f"  - Análise de vendas baseada em um período de {periodo_dias} dias.")

df_analise_completa = pd.merge(vendas_agregado, df_analise_curva[['chave_vendas', 'curva']], on='chave_vendas', how='left')

if chave_vendas_col == 'cod_produto':
    df_final['chave_vendas'] = df_final['product_code'].astype(str).str.strip().str.upper()
else:
    df_final['chave_vendas'] = df_final['product_name'].astype(str).str.strip().str.upper()

df_final = pd.merge(df_final, df_analise_completa[['chave_vendas', 'venda_total', 'venda_media_diaria', 'curva']], on='chave_vendas', how='left')

df_final[['venda_total', 'venda_media_diaria']] = df_final[['venda_total', 'venda_media_diaria']].fillna(0)

# Produtos sem vendas ou venda_total ausente ficam como curva D
if 'curva' in df_final.columns:
    if str(df_final['curva'].dtype) == 'category' and 'D' not in df_final['curva'].cat.categories:
        df_final['curva'] = df_final['curva'].cat.add_categories(['D'])
    mask_sem_vendas = df_final['venda_total'].isnull() | (df_final['venda_total'] <= 0)
    df_final.loc[mask_sem_vendas, 'curva'] = 'D'
    df_final['curva'].fillna('C', inplace=True)

print("✅ Análise de vendas concluída e integrada.")


# ==============================================================================
# Bloco 7: Cálculos Finais e Formatação
# ==============================================================================
print("\n--- [ Etapa 5 de 6 ]: Realizando Cálculos Finais e Formatando ---")

# --- 7a. Cálculo de Volume em Litros e Dias de Estoque ---

df_final['vol_L_unitario'] = df_final['volume_cm3_final'] / 1000
df_final['vol_L_total_unitario'] = df_final['vol_L_unitario'] * df_final['quantidade']
df_final['vol_L_total'] = df_final['vol_L_total_unitario']

# --- 7a.1 Tipo de equipamento base (sempre) ---
def _categoria_para_tipo(cat):
    s = str(cat or '').strip().lower()
    if 'refrigerado' in s or 'geladeira' in s:
        return 'geladeira'
    if 'congelado' in s or 'freezer' in s:
        return 'freezer'
    return 'prateleira'

df_final['tipo_equipamento_base'] = df_final['categoria_armazenagem'].apply(_categoria_para_tipo)

# --- 7a.2 Logica de caixaria (opcional) ---
if 'caixaria' not in df_final.columns:
    df_final['caixaria'] = np.nan
for col in ['caixa_largura_cm', 'caixa_altura_cm', 'caixa_comprimento_cm', 'caixa_volume_cm3']:
    if col not in df_final.columns:
        df_final[col] = np.nan

def _calc_volume_caixa_cm3(row):
    v = _to_float(row.get('caixa_volume_cm3'))
    if v is not None and v > 0:
        return v
    l = _to_float(row.get('caixa_largura_cm'))
    a = _to_float(row.get('caixa_altura_cm'))
    c = _to_float(row.get('caixa_comprimento_cm'))
    if l and a and c and l > 0 and a > 0 and c > 0:
        return l * a * c
    return None

df_final['caixa_volume_cm3_final'] = df_final.apply(_calc_volume_caixa_cm3, axis=1)
df_final['qtd_em_caixas'] = np.where(
    df_final['caixaria'].notna() & (df_final['caixaria'] > 0),
    df_final['quantidade'] / df_final['caixaria'],
    np.nan
)

def _decidir_metodo_caixa(row):
    tipo = str(row.get('tipo_equipamento_base', '')).strip().lower().replace(' ', '_')
    if 'geladeira' in tipo or 'freezer' in tipo:
        return 'unitario'
    caixaria = _to_float(row.get('caixaria'))
    vol_caixa = _to_float(row.get('caixa_volume_cm3_final'))
    qtd = row.get('qtd_em_caixas')
    if caixaria is None or caixaria <= 0 or vol_caixa is None or vol_caixa <= 0:
        return 'verificar'
    if qtd is None or (isinstance(qtd, float) and np.isnan(qtd)):
        return 'verificar'
    if qtd > 0.5:
        return 'caixa'
    if qtd < 0.5:
        return 'unitario'
    return 'unitario'

df_final['metodo'] = df_final.apply(_decidir_metodo_caixa, axis=1)

def _arredondar_caixas(qtd):
    if qtd is None or (isinstance(qtd, float) and np.isnan(qtd)):
        return None
    base = int(np.floor(qtd))
    frac = qtd - base
    if frac >= 0.3:
        return base + 1
    return base

df_final['caixas_necessarias'] = df_final.apply(
    lambda r: _arredondar_caixas(r.get('qtd_em_caixas')) if r.get('metodo') == 'caixa' else np.nan,
    axis=1
)

mask_caixa = (
    (df_final['metodo'] == 'caixa')
    & df_final['caixas_necessarias'].notna()
    & (df_final['caixas_necessarias'] > 0)
    & df_final['caixa_volume_cm3_final'].notna()
    & (df_final['caixa_volume_cm3_final'] > 0)
)
df_final.loc[mask_caixa, 'vol_L_total'] = (
    (df_final.loc[mask_caixa, 'caixa_volume_cm3_final'] / 1000)
    * df_final.loc[mask_caixa, 'caixas_necessarias']
)

# --- 7a.1 Calculo de escaninhos necessarios ---
capacidade_por_tipo = {}
if 'df_vol_equip' in locals() and df_vol_equip is not None and not df_vol_equip.empty and 'tipo_equipamento' in df_vol_equip.columns:
    for _, row in df_vol_equip.iterrows():
        tipo = _normalizar_tipo_equip(row.get('tipo_equipamento', ''))
        if not tipo or tipo == 'tipo_equipamento':
            continue
        l = _to_float(row.get('l_por_escaninho'))
        f = _to_float(row.get('fator_seguranca'))
        if l and f and l > 0 and f > 0:
            capacidade_por_tipo[tipo] = l * f

if capacidade_por_tipo:
    df_final['escaninhos_necessarios'] = df_final.apply(
        lambda r: _calc_escaninhos(r.get('vol_L_total', 0), capacidade_por_tipo.get(_normalizar_tipo_equip(r.get('tipo_equipamento_base', '')))),
        axis=1
    )

    # Regra de negocio: Geladeira + degelo PODE => escaninhos_necessarios max 4
    if 'categoria_armazenagem' in df_final.columns and 'degelo' in df_final.columns:
        mask_limite_escaninhos = (
            df_final['categoria_armazenagem'].astype(str).str.strip().str.lower().eq('geladeira')
            & df_final['degelo'].astype(str).str.strip().str.upper().str.startswith('PODE')
        )
        if mask_limite_escaninhos.any():
            df_final.loc[mask_limite_escaninhos, 'escaninhos_necessarios'] = (
                df_final.loc[mask_limite_escaninhos, 'escaninhos_necessarios'].clip(upper=4)
            )
            print(f"  - Regra aplicada: {int(mask_limite_escaninhos.sum())} produtos Geladeira com degelo PODE limitados a 4 escaninhos.")
    else:
        print("  - [AVISO] Colunas 'categoria_armazenagem' ou 'degelo' nao encontradas para limitar escaninhos.")

    print(f"  - Escaninhos necessarios calculados para {len(capacidade_por_tipo)} tipo(s) de equipamento.")
else:
    df_final['escaninhos_necessarios'] = 1
    print("WARNING: Nao foi possivel calcular escaninhos necessarios (volumetria de equipamentos ausente).")


df_final['dias_estoque'] = np.divide(
    df_final['quantidade'],
    df_final['venda_media_diaria'],
    out=np.zeros_like(df_final['quantidade'], dtype=float),
    where=df_final['venda_media_diaria'] != 0
)
print("  - Métricas de volume e dias de estoque calculadas.")

# --- 7b. Arredondamento para 2 casas decimais ---
df_final['venda_media_diaria'] = df_final['venda_media_diaria'].round(2)
df_final['dias_estoque'] = df_final['dias_estoque'].round(2)
print("  - Colunas de vendas e dias de estoque arredondadas para 2 casas decimais.")

# --- 7c. Seleção e Ordenação Final de Colunas ---
colunas_finais_solicitadas = [
    'product_code', 'product_name', 'nm_fabricante', 'categoria_armazenagem',
    'subcategoria',
    'categoria_site', 'altura_cm', 'vol_L_unitario', 'quantidade', 'curva',
    'largura_cm', 'comprimento_cm', 'vol_L_total', 'venda_total',
    'venda_media_diaria', 'dias_estoque',
    'peso_kg_unitario',
    'is_pesado',
    'is_fragil', 'degelo', 'prioridade_alocacao',
    'caixaria', 'qtd_em_caixas', 'metodo', 'caixa_volume_cm3_final', 'caixas_necessarias'
]

# Adiciona colunas de escaninhos necessarios (se existirem)
extras_escaninhos = ['escaninhos_necessarios', 'tipo_equipamento_base']
for c in extras_escaninhos:
    if c not in colunas_finais_solicitadas:
        colunas_finais_solicitadas.append(c)

# Mantém só as colunas presentes
colunas_finais = [c for c in colunas_finais_solicitadas if c in df_final.columns]
df_final_formatado = df_final[colunas_finais].copy()
print("✅ Colunas finais selecionadas e reordenadas conforme solicitado.")


# ==============================================================================
# Bloco 8: Extração de Peso e Geração de Log Consolidado
# ==============================================================================
print("\n--- [ Etapa 6 de 6 ]: Extraindo peso dos produtos ---")

# DICIONÁRIO MÍNIMO: Apenas para exceções que NÃO possuem peso explícito no nome.
DICIONARIO_EXCECOES_PESADAS = {
    'MELANCIA': 4.0, 'ABACAXI': 1.8, 'COCO': 1.5, 'MELÃO': 1.8
}

def extrair_peso_do_nome_v8(product_name):
    if not isinstance(product_name, str):
        return None, "ERRO_NOME_INVALIDO"
    name_upper = product_name.upper()

    # Regra 0: Exceções que podem ser confundidas (ex: Saco de Lixo com 'L' de Litro)
    if 'SACO DE LIXO' in name_upper or 'SACO PARA CONSERVAR' in name_upper:
        return 0.300, "EXCECAO_SACO_LIXO"

    # Regra 1: Regra especial para Ovos
    if 'OVO' in name_upper:
        match_qtd = re.search(r'(?:C/|COM)\s*(\d+)', name_upper)
        if match_qtd:
            quantidade = int(match_qtd.group(1))
            if 'CODORNA' in name_upper:
                return quantidade * 0.010, f"CALCULO_OVO_CODORNA_{quantidade}UN"
            return quantidade * 0.055, f"CALCULO_OVO_GALINHA_{quantidade}UN"

    # Regra 2: Tentar extrair peso explícito
    match_per_unit = re.search(r'(?:C/|COM)\s*(\d+)\s*UN(?:IDADES)?\s*DE\s*(\d[\d,.]*)\s*(G|KG|L|ML)', name_upper)
    if match_per_unit:
        qtd, valor_str, unidade = match_per_unit.groups()
        valor = float(valor_str.replace(',', '.'))
        peso_unit_g = valor if unidade in ['G', 'ML'] else valor * 1000
        return (int(qtd) * peso_unit_g) / 1000, f"PACK_POR_UNIDADE_{unidade}"

    match_total = re.search(r'(\d[\d,.]*)\s*(KG|G|L|ML)', name_upper)
    if match_total:
        valor_str, unidade = match_total.groups()
        valor = float(valor_str.replace(',', '.'))
        return valor if unidade in ['KG', 'L'] else valor / 1000, f"EXPLICITO_TOTAL_{unidade}"

    # Regra 3: Se NENHUM peso explícito foi encontrado, consultar o dicionário.
    for termo, peso in DICIONARIO_EXCECOES_PESADAS.items():
        if termo in name_upper:
            return peso, f"DICIONARIO_EXCECAO_{termo}"

    # Regra 4: Default para o resto
    return None, "NAO_ENCONTRADO"

# Aplicação e geração de logs
if 'product_name' not in df_final_formatado.columns:
    print("\n❌ ERRO CRÍTICO: A coluna 'product_name' é obrigatória para extração de peso.")
    sys.exit()

logs_extracao = []
PESO_LEVE_PADRAO = 0.300

for index, row in df_final_formatado.iterrows():
    peso, regra = extrair_peso_do_nome_v8(row['product_name'])
    log_peso, log_regra = (peso, regra)

    if peso is None:
        log_regra, log_peso = "DEFAULT_LEVE", PESO_LEVE_PADRAO
        df_final_formatado.loc[index, 'peso_kg_unitario'] = PESO_LEVE_PADRAO
    else:
        df_final_formatado.loc[index, 'peso_kg_unitario'] = peso

    logs_extracao.append({
        'product_code': row.get('product_code', 'N/A'),
        'product_name': row['product_name'],
        'regra_aplicada': log_regra,
        'peso_kg_calculado': log_peso
    })

if 'quantidade' in df_final_formatado.columns:
    df_final_formatado['peso_kg_total'] = df_final_formatado['peso_kg_unitario'] * df_final_formatado['quantidade']

# Classificacao de pesado a partir do limite configurado
limite_peso_kg_utilizado = limite_peso_kg if limite_peso_kg is not None else 0.7
if 'peso_kg_unitario' in df_final_formatado.columns:
    peso_num = pd.to_numeric(df_final_formatado['peso_kg_unitario'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)
    df_final_formatado['is_pesado'] = peso_num >= limite_peso_kg_utilizado
    print(f"  - is_pesado calculado com limite_peso_kg = {limite_peso_kg_utilizado} kg.")

# Log consolidado
df_logs_peso = pd.DataFrame(logs_extracao)
print("✅ Processamento de peso e geração de logs concluídos.")


# ==============================================================================
# ==============================================================================
# Bloco 9: Escrita de Resultados
# ==============================================================================
print("\n--- [ Final ]: Gravando resultados ---")

# Abas geradas pelo script
ABAS_SAIDA = [
    NOME_ABA_SAIDA_MIX,
    'Log_Extracao_Peso',
    'Log_Produtos_Sem_Dados',
    'Log_Produtos_Sem_Subcategoria',
    'Log_Produtos_Sem_Categoria_Arm',
    'Log_Geladeira_Degelo_Vazio'
]

if USE_LOCAL_XLSX:
    output_path = LOCAL_OUTPUT_XLSX
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_final_formatado.to_excel(writer, sheet_name=NOME_ABA_SAIDA_MIX, index=False)
        if not df_logs_peso.empty:
            df_logs_peso.to_excel(writer, sheet_name='Log_Extracao_Peso', index=False)
        if not df_log_volumetria.empty:
            df_log_volumetria.to_excel(writer, sheet_name='Log_Produtos_Sem_Dados', index=False)
        if not df_log_subcategoria.empty:
            df_log_subcategoria.to_excel(writer, sheet_name='Log_Produtos_Sem_Subcategoria', index=False)
        if not df_log_categoria_armazenagem.empty:
            df_log_categoria_armazenagem.to_excel(writer, sheet_name='Log_Produtos_Sem_Categoria_Arm', index=False)
        if not df_log_degelo_geladeira.empty:
            df_log_degelo_geladeira.to_excel(writer, sheet_name='Log_Geladeira_Degelo_Vazio', index=False)
    print(f"  - Arquivo local gerado: '{output_path}'.")
else:
    def _resetar_abas_saida(spreadsheet_obj, abas):
        titulos = [ws.title for ws in spreadsheet_obj.worksheets()]
        for nome in abas:
            if nome in titulos:
                try:
                    ws = spreadsheet_obj.worksheet(nome)
                    spreadsheet_obj.del_worksheet(ws)
                    print(f"  - Aba '{nome}' removida.")
                except Exception:
                    try:
                        ws = spreadsheet_obj.worksheet(nome)
                        ws.clear()
                        print(f"  - Aba '{nome}' limpa (fallback).")
                    except Exception as e2:
                        print(f"  - [AVISO] Não foi possível limpar/remover a aba '{nome}': {e2}")

    # Limpa/remove as abas de saída antes de recriar
    _resetar_abas_saida(spreadsheet, ABAS_SAIDA)

    # Função para escrever DataFrame na aba (cria ou atualiza)
    def escrever_aba(spreadsheet_obj, nome_aba, df):
        titulos = [ws.title for ws in spreadsheet_obj.worksheets()]
        if nome_aba in titulos:
            ws = spreadsheet_obj.worksheet(nome_aba)
            ws.clear()
        else:
            rows = max(len(df) + 1, 1)
            cols = max(len(df.columns), 1)
            ws = spreadsheet_obj.add_worksheet(title=nome_aba, rows=rows, cols=cols)
        gd.set_with_dataframe(ws, df, include_index=False, include_column_header=True, resize=True)

    # Aba principal
    escrever_aba(spreadsheet, NOME_ABA_SAIDA_MIX, df_final_formatado)
    print(f"  - Aba '{NOME_ABA_SAIDA_MIX}' atualizada com {len(df_final_formatado)} linhas.")

    # Abas de log (somente se houver dados)
    if not df_logs_peso.empty:
        escrever_aba(spreadsheet, 'Log_Extracao_Peso', df_logs_peso)
        print("  - Log_Extracao_Peso gravado.")
    if not df_log_volumetria.empty:
        escrever_aba(spreadsheet, 'Log_Produtos_Sem_Dados', df_log_volumetria)
        print("  - Log_Produtos_Sem_Dados gravado.")
    if not df_log_subcategoria.empty:
        escrever_aba(spreadsheet, 'Log_Produtos_Sem_Subcategoria', df_log_subcategoria)
        print("  - Log_Produtos_Sem_Subcategoria gravado.")
    if not df_log_categoria_armazenagem.empty:
        escrever_aba(spreadsheet, 'Log_Produtos_Sem_Categoria_Arm', df_log_categoria_armazenagem)
        print("  - Log_Produtos_Sem_Categoria_Arm gravado.")
    if not df_log_degelo_geladeira.empty:
        escrever_aba(spreadsheet, 'Log_Geladeira_Degelo_Vazio', df_log_degelo_geladeira)
        print("  - Log_Geladeira_Degelo_Vazio gravado.")

print("✅ Processo finalizado com sucesso!")

